# Advanced: Multi-Strategy Attribute Profiling with PAMOLA.CORE

**Goal**: Master all 3 attribute profiling strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Default semantic + statistical profiling
- **Strategy 2**: Statistical-only profiling (entropy-based)
- **Strategy 3**: Custom dictionary with domain-specific keywords
- Calculate advanced privacy metrics (k-anonymity)
- Analyze re-identification risk
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts (identifiers, quasi-identifiers)
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_data_attribute_profiler_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.attribute import DataAttributeProfilerOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 20 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique values)
- Mix of identifiers, demographics, healthcare, and financial data

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'attribute_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic healthcare/employee dataset
    first_names = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Henry', 'Iris', 'Jack'] * 100
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez'] * 100
    
    df = pd.DataFrame({
        # Direct identifiers
        'employee_id': [f'EMP{i:06d}' for i in range(1, n_records + 1)],
        'email_address': [f'{first_names[i].lower()}.{last_names[i].lower()}{i}@company.com' for i in range(n_records)],
        'phone_number': [f'+1-555-{np.random.randint(100, 999)}-{np.random.randint(1000, 9999)}' for _ in range(n_records)],
        'ssn': [f'{np.random.randint(100, 999)}-{np.random.randint(10, 99)}-{np.random.randint(1000, 9999)}' for _ in range(n_records)],
        'first_name': first_names,
        'last_name': last_names,
        
        # Quasi-identifiers
        'date_of_birth': pd.date_range('1960-01-01', periods=n_records, freq='5D'),
        'gender': np.random.choice(['Male', 'Female', 'Non-binary'], n_records, p=[0.48, 0.48, 0.04]),
        'postal_code': np.random.choice([f'{i:05d}' for i in range(10000, 10050)], n_records),
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia'], n_records),
        'state': np.random.choice(['NY', 'CA', 'IL', 'TX', 'AZ', 'PA'], n_records),
        'job_title': np.random.choice(['Engineer', 'Manager', 'Analyst', 'Consultant', 'Specialist', 'Director'], n_records),
        'department': np.random.choice(['IT', 'HR', 'Finance', 'Sales', 'Marketing', 'Operations'], n_records),
        'years_experience': np.random.randint(0, 40, n_records),
        
        # Sensitive attributes
        'annual_salary': np.random.lognormal(mean=11.0, sigma=0.5, size=n_records).astype(int),
        'ethnicity': np.random.choice(['Asian', 'Caucasian', 'African American', 'Hispanic', 'Other'], n_records),
        'health_condition': np.random.choice(['None', 'Diabetes', 'Hypertension', 'Asthma', 'Arthritis'], n_records, p=[0.6, 0.1, 0.1, 0.1, 0.1]),
        'credit_score': np.random.randint(300, 850, n_records),
        
        # Indirect identifiers
        'device_id': [f'DEV-{np.random.randint(100000, 999999)}' for _ in range(n_records)],
        
        # Non-sensitive
        'hire_date': pd.date_range('2010-01-01', periods=n_records, freq='8h'),
    })
    
    # Add some nulls for realism
    null_indices = np.random.choice(df.index, size=int(n_records * 0.05), replace=False)
    for col in ['phone_number', 'health_condition', 'credit_score']:
        df.loc[null_indices[:len(null_indices)//3], col] = np.nan
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns[:5])}... ({len(df.columns)} total)")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns[:10]:  # Show first 10
    dtype_str = str(df[col].dtype)
    
    # Handle different data types safely
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == 'object':
        # For string/object columns, just show unique values
        nunique = df[col].nunique()
        print(f"  {col:20s} ({dtype_str:10s}): {nunique} unique values")
        
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        # For datetime columns, show date range
        try:
            min_val = df[col].min()
            max_val = df[col].max()
            print(f"  {col:20s} ({dtype_str:10s}): {min_val} to {max_val}")
        except:
            print(f"  {col:20s} ({dtype_str:10s}): [unable to compute range]")
            
    elif pd.api.types.is_numeric_dtype(df[col]):
        # For numeric columns, show min/max (skip NaN values)
        try:
            min_val = df[col].min()
            max_val = df[col].max()
            
            # Format based on type
            if pd.api.types.is_integer_dtype(df[col]):
                print(f"  {col:20s} ({dtype_str:10s}): min={min_val}, max={max_val}")
            else:
                print(f"  {col:20s} ({dtype_str:10s}): min={min_val:.2f}, max={max_val:.2f}")
        except:
            print(f"  {col:20s} ({dtype_str:10s}): [unable to compute range]")
    else:
        # Fallback for other types
        print(f"  {col:20s} ({dtype_str:10s}): [mixed type]")

if len(df.columns) > 10:
    print(f"  ... and {len(df.columns) - 10} more columns")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Custom Dictionary

**How to use:**
- Run to check/create custom attribute dictionary
- Uses existing file if found
- Creates domain-specific example if missing

**What you'll see:**
- File status (✓ found or 🔧 created)
- File location path
- Dictionary structure (5 categories, keywords count)
- Example keywords per category

**Note:** Edit file to match your domain-specific attributes before Strategy 3

In [ ]:
# Define custom dictionary path (in examples directory)
examples_dir = project_root / 'examples'
custom_dict_path = examples_dir / 'data_examples' / 'custom_attribute_dictionary.json'

# Check if file already exists
if custom_dict_path.exists():
    print(f"✓ Found existing custom dictionary: {custom_dict_path}")
    print(f"📂 Using existing file for Strategy 3\n")
    
    # Load and display info
    with open(custom_dict_path, 'r') as f:
        existing_dict = json.load(f)
    
    print("📊 Existing Dictionary Info:")
    categories = existing_dict.get('categories', {})
    print(f"  Categories: {len(categories)}")
    for cat_name in categories:
        print(f"    • {cat_name}")
    
    print("\n💡 To use a different dictionary, replace or rename it.")
    print("=" * 80)

else:
    print(f"⚠️  Custom dictionary not found!")
    print(f"🔧 Creating example custom dictionary...\n")
    print("=" * 80)

    # Define custom dictionary with extended keywords
    custom_dict = {
        "categories": {
            "DIRECT_IDENTIFIER": {
                "description": "Explicit identifiers - unique and directly identify individuals",
                "keywords": [
                    "id", "uuid", "uid", "guid", "identifier", "employee_id", "patient_id",
                    "email", "e-mail", "mail", "email_address",
                    "phone", "telephone", "mobile", "cell", "phone_number",
                    "ssn", "sin", "passport", "driver_license", "national_id",
                    "first_name", "last_name", "full_name", "surname", "given_name"
                ]
            },
            "QUASI_IDENTIFIER": {
                "description": "Attributes that allow identification in combination",
                "keywords": [
                    "address", "street", "postal_code", "zip", "zipcode",
                    "city", "town", "municipality", "locality",
                    "state", "province", "region", "district",
                    "country", "nation",
                    "birth", "dob", "date_of_birth", "birth_date", "birthdate",
                    "age", "age_group",
                    "gender", "sex",
                    "job", "position", "job_title", "occupation", "profession",
                    "company", "employer", "organization", "department",
                    "education", "degree", "university", "school",
                    "experience", "years_experience", "work_history"
                ]
            },
            "SENSITIVE_ATTRIBUTE": {
                "description": "Confidential or sensitive information",
                "keywords": [
                    "ethnicity", "race", "nationality",
                    "religion", "faith", "belief",
                    "salary", "income", "earnings", "annual_salary", "compensation", "pay", "wage",
                    "diagnosis", "disease", "health", "medical", "health_condition", "illness",
                    "treatment", "medication", "therapy", "prescription",
                    "bank", "account", "credit_card", "credit_score", "financial",
                    "balance", "loan", "debt", "mortgage",
                    "political", "vote", "opinion", "preference"
                ]
            },
            "INDIRECT_IDENTIFIER": {
                "description": "Potentially identifying through content analysis",
                "keywords": [
                    "geo", "longitude", "latitude", "coordinates", "gps", "location",
                    "ip", "ip_address", "mac", "mac_address",
                    "device", "device_id", "hardware_id", "imei",
                    "browser", "user_agent", "fingerprint",
                    "photo", "image", "picture", "avatar",
                    "biometric", "fingerprint", "facial", "voice",
                    "notes", "comments", "description", "bio", "profile"
                ]
            },
            "NON_SENSITIVE": {
                "description": "Non-identifying, non-sensitive information",
                "keywords": [
                    "timestamp", "date", "datetime", "created", "updated",
                    "status", "state", "stage", "active", "inactive",
                    "event", "log", "action", "activity",
                    "tag", "label", "category", "type", "class",
                    "count", "quantity", "number", "total",
                    "flag", "indicator", "marker"
                ]
            }
        },
        "statistical_thresholds": {
            "entropy_high": 5.0,
            "entropy_mid": 3.0,
            "uniqueness_high": 0.9,
            "uniqueness_low": 0.2,
            "text_length_long": 100,
            "mvf_threshold": 2
        }
    }
    
    # Save to file (with error handling)
    try:
        custom_dict_path.parent.mkdir(parents=True, exist_ok=True)
        with open(custom_dict_path, 'w') as f:
            json.dump(custom_dict, f, indent=2)
        print(f"✓ Example custom dictionary created: {custom_dict_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {custom_dict_path} (file may be open)")
        print(f"   Dictionary exists in memory only")

    print(f"\n📊 Dictionary Statistics:")
    categories = custom_dict['categories']
    for cat_name, cat_data in categories.items():
        keyword_count = len(cat_data.get('keywords', []))
        print(f"  {cat_name:25s}: {keyword_count} keywords")

    print(f"\n📋 Example Keywords:")
    for cat_name, cat_data in list(categories.items())[:2]:  # Show first 2
        keywords = cat_data.get('keywords', [])
        print(f"  {cat_name}:")
        print(f"    {', '.join(keywords[:5])}...")

    print("\n💡 You can edit this file to match your domain!")
    print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
- Run to setup analysis environment
- No field customization needed (analyzes ALL columns)
- Validates dataset and creates task directory

**What you'll see:**
- Dataset validation (✓ 1000 records, 20 columns)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs ready (✓)
- DataSource created (✓)

**Note:** This operation automatically analyzes all columns - no manual selection required

In [ ]:
# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Validate dataset
print("📋 Validating dataset...\n")
print(f"  ✓ Records: {len(df):,}")
print(f"  ✓ Columns: {len(df.columns)}")
print(f"  ✓ All columns will be analyzed automatically")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy attribute profiling",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Default Semantic + Statistical

**How to use:**
- Uses default built-in dictionary
- Combines semantic (keyword) and statistical analysis
- Run to execute default strategy

**Key parameters:**
- `dictionary_path=None`: Use built-in default dictionary
- `language='en'`: English keyword matching
- `sample_size=10`: Sample values per column
- Analyzes ALL columns automatically

**What you'll see:**
- Configuration summary
- Progress: validation → analysis → categorization → metrics → save
- Completion time (2-5 seconds)
- Attribute role distribution (counts per category)

**Note:** Best for general-purpose profiling with standard attribute names

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: DEFAULT SEMANTIC + STATISTICAL")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Default",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = DataAttributeProfilerOperation(
    name='AttributeProfiler_Default',
    dictionary_path=None,  # Use default dictionary
    language='en',
    sample_size=10,
    max_columns=None,  # Analyze all columns
    use_vectorization=False,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_default',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_default' / 'output').glob('attribute_roles_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    with open(output_files_s1[0], 'r') as f:
        roles_s1 = json.load(f)
    
    summary = roles_s1.get('summary', {})
    print(f"\n📊 Role Distribution:")
    for role, count in summary.items():
        print(f"   {role:25s}: {count}")

## STRATEGY 2: Statistical-Only Analysis

**How to use:**
- Ignores column names (semantic analysis disabled)
- Uses only statistical properties (entropy, uniqueness)
- Run to execute statistical-only strategy

**Key parameters:**
- Empty dictionary simulates statistical-only mode
- `sample_size=15`: More samples for better statistics
- Relies on entropy and uniqueness thresholds

**What you'll see:**
- Configuration confirmation
- Progress: validation → statistical analysis → categorization → save
- Completion time (2-5 seconds)
- Different role distribution (statistical patterns only)

**Note:** Useful when column names are non-descriptive or in foreign languages

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: STATISTICAL-ONLY ANALYSIS")
print("=" * 80 + "\n")

# Create minimal dictionary (statistical-only mode)
stat_only_dict_path = task_dir / 'stat_only_dict.json'
stat_only_dict = {
    "categories": {
        "DIRECT_IDENTIFIER": {"description": "Direct identifiers", "keywords": []},
        "QUASI_IDENTIFIER": {"description": "Quasi-identifiers", "keywords": []},
        "SENSITIVE_ATTRIBUTE": {"description": "Sensitive attributes", "keywords": []},
        "INDIRECT_IDENTIFIER": {"description": "Indirect identifiers", "keywords": []},
        "NON_SENSITIVE": {"description": "Non-sensitive", "keywords": []}
    },
    "statistical_thresholds": {
        "entropy_high": 5.0,
        "entropy_mid": 3.0,
        "uniqueness_high": 0.9,
        "uniqueness_low": 0.2
    }
}

with open(stat_only_dict_path, 'w') as f:
    json.dump(stat_only_dict, f, indent=2)

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Statistical",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = DataAttributeProfilerOperation(
    name='AttributeProfiler_Statistical',
    dictionary_path=str(stat_only_dict_path),
    language='en',
    sample_size=15,
    max_columns=None,
    use_vectorization=False,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured (statistical-only mode)")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_statistical',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_statistical' / 'output').glob('attribute_roles_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    with open(output_files_s2[0], 'r') as f:
        roles_s2 = json.load(f)
    
    summary = roles_s2.get('summary', {})
    print(f"\n📊 Role Distribution:")
    for role, count in summary.items():
        print(f"   {role:25s}: {count}")

## STRATEGY 3: Custom Domain Dictionary

**How to use:**
- Uses custom dictionary from Step 3
- Extended keywords for specific domain
- Run to execute custom dictionary strategy

**Key parameters:**
- `dictionary_path=custom_dict_path`: Use custom dictionary
- Enhanced keyword matching with domain terms
- `sample_size=10`: Standard sampling

**What you'll see:**
- Configuration confirmation
- Progress: validation → custom analysis → categorization → save
- Completion time (2-5 seconds)
- More accurate role assignments (domain-specific keywords)

**Note:** Best for domain-specific datasets (healthcare, finance, HR)

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: CUSTOM DOMAIN DICTIONARY")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Custom",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = DataAttributeProfilerOperation(
    name='AttributeProfiler_Custom',
    dictionary_path=str(custom_dict_path),
    language='en',
    sample_size=10,
    max_columns=None,
    use_vectorization=False,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_custom',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_custom' / 'output').glob('attribute_roles_*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    with open(output_files_s3[0], 'r') as f:
        roles_s3 = json.load(f)
    
    summary = roles_s3.get('summary', {})
    print(f"\n📊 Role Distribution:")
    for role, count in summary.items():
        print(f"   {role:25s}: {count}")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and accuracy

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Role distribution comparison (counts per category)
- Accuracy differences between strategies

**Note:** Custom dictionary typically most accurate, statistical-only fastest but less precise

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Role Distribution Comparison:")
if output_files_s1 and output_files_s2 and output_files_s3:
    summary_s1 = roles_s1.get('summary', {})
    summary_s2 = roles_s2.get('summary', {})
    summary_s3 = roles_s3.get('summary', {})
    
    role_names = ['DIRECT_IDENTIFIER', 'QUASI_IDENTIFIER', 'SENSITIVE_ATTRIBUTE', 
                  'INDIRECT_IDENTIFIER', 'NON_SENSITIVE']
    
    print(f"\n{'Role':<25} {'Strategy 1':>12} {'Strategy 2':>12} {'Strategy 3':>12}")
    print("-" * 70)
    for role in role_names:
        s1_count = summary_s1.get(role, 0)
        s2_count = summary_s2.get(role, 0)
        s3_count = summary_s3.get(role, 0)
        print(f"{role:<25} {s1_count:>12} {s2_count:>12} {s3_count:>12}")
    
    print(f"\n💡 Key Differences:")
    print(f"   • Strategy 1 (Default): Balanced approach")
    print(f"   • Strategy 2 (Statistical): Pattern-based, may over-classify")
    print(f"   • Strategy 3 (Custom): Domain-specific, typically most accurate")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Performance and categorization metrics
2. **Attribute Roles**: Detailed categorization with confidence scores
3. **Entropy Analysis**: Statistical properties per column
4. **Visualizations**: Charts (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_default', 'Strategy 1: Default'),
    ('strategy2_statistical', 'Strategy 2: Statistical'),
    ('strategy3_custom', 'Strategy 3: Custom')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                print(f"   Total columns: {metrics.get('total_columns', 0)}")
                print(f"   Direct IDs: {metrics.get('direct_identifiers_count', 0)}")
                print(f"   Quasi-IDs: {metrics.get('quasi_identifiers_count', 0)}")
                print(f"   Sensitive: {metrics.get('sensitive_attributes_count', 0)}")
                print(f"   Avg entropy: {metrics.get('avg_entropy', 0):.4f}")
                print(f"   Avg uniqueness: {metrics.get('avg_uniqueness', 0):.4f}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Privacy Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates k-anonymity using quasi-identifiers

**What you'll see:**
- Quasi-identifiers identified per strategy
- K-anonymity calculation (min k, avg k, vulnerable groups)
- Comparison across strategies

**Privacy targets:**
- Min k ≥ 5: Basic protection
- Min k ≥ 10: Strong protection
- Zero vulnerable groups: Ideal

**Note:** Different strategies may identify different quasi-identifiers, affecting k-anonymity

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS ANALYSIS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics for given quasi-identifiers."""
    if not quasi_identifiers:
        return None
    
    # Filter to columns that exist
    valid_qis = [qi for qi in quasi_identifiers if qi in df.columns]
    if not valid_qis:
        return None
    
    group_sizes = df.groupby(valid_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum()),
        'quasi_identifiers': valid_qis
    }

# Calculate for each strategy
strategies = [
    ('Strategy 1 (Default)', roles_s1),
    ('Strategy 2 (Statistical)', roles_s2),
    ('Strategy 3 (Custom)', roles_s3)
]

for name, roles_data in strategies:
    print(f"📊 {name}:")
    print("-" * 60)
    
    quasi_ids = roles_data.get('column_groups', {}).get('QUASI_IDENTIFIER', [])
    print(f"   Quasi-identifiers: {len(quasi_ids)}")
    if quasi_ids:
        print(f"   Fields: {', '.join(quasi_ids[:5])}{'...' if len(quasi_ids) > 5 else ''}")
    
    k_metrics = calculate_k_anonymity(df, quasi_ids)
    if k_metrics:
        print(f"   Min k: {k_metrics['min_k']}")
        print(f"   Avg k: {k_metrics['avg_k']:.2f}")
        print(f"   Vulnerable groups: {k_metrics['vulnerable_groups']}")
        
        if k_metrics['min_k'] >= 10:
            print(f"   ✅ Strong privacy protection (k≥10)")
        elif k_metrics['min_k'] >= 5:
            print(f"   ⚠️  Basic privacy protection (5≤k<10)")
        else:
            print(f"   ❌ Weak privacy protection (k<5)")
    else:
        print(f"   ℹ️  Cannot calculate (no valid quasi-identifiers)")
    
    print()

print("💡 Lower min k = higher re-identification risk!")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports analysis results for each strategy

**What you'll see (per strategy):**
- Attribute roles file location
- Entropy analysis file location
- Key metrics summary

**Output structure:**
```
advanced_output/
├── strategy1_default/
│   └── output/
│       ├── attribute_roles_*.json
│       └── attribute_entropy_*.csv
├── strategy2_statistical/
│   └── output/
│       ├── attribute_roles_*.json
│       └── attribute_entropy_*.csv
└── strategy3_custom/
    └── output/
        ├── attribute_roles_*.json
        └── attribute_entropy_*.csv
```

**Note:** Files include complete analysis for comparison and further processing

In [ ]:
print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
print(f"\n📂 Export directory: {task_dir}\n")

for dir_name, title in strategy_dirs:
    strategy_dir = export_base_dir / dir_name
    if not strategy_dir.exists():
        continue
    
    print("=" * 80)
    print(f"📊 {title.upper()}")
    print("=" * 80)
    
    output_dir = strategy_dir / 'output'
    if not output_dir.exists():
        print("⚠️  Output directory not found\n")
        continue
    
    # List key files
    roles_files = list(output_dir.glob('attribute_roles_*.json'))
    entropy_files = list(output_dir.glob('attribute_entropy_*.csv'))
    sample_files = list(output_dir.glob('attribute_sample_*.json'))
    
    print(f"\n📄 Generated Files:")
    if roles_files:
        print(f"   ✓ Attribute roles: {roles_files[0].name}")
        print(f"     Location: {roles_files[0]}")
    
    if entropy_files:
        print(f"   ✓ Entropy analysis: {entropy_files[0].name}")
        print(f"     Location: {entropy_files[0]}")
    
    if sample_files:
        print(f"   ✓ Sample values: {sample_files[0].name}")
        print(f"     Location: {sample_files[0]}")
    
    # Load and show summary
    if roles_files:
        with open(roles_files[0], 'r') as f:
            roles = json.load(f)
        
        summary = roles.get('summary', {})
        print(f"\n📈 Summary:")
        print(f"   Total columns: {sum(summary.values())}")
        print(f"   Direct IDs: {summary.get('DIRECT_IDENTIFIER', 0)}")
        print(f"   Quasi-IDs: {summary.get('QUASI_IDENTIFIER', 0)}")
        print(f"   Sensitive: {summary.get('SENSITIVE_ATTRIBUTE', 0)}")
    
    print()

# ============================================================================
# SUMMARY
# ============================================================================
print("=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 attribute profiling strategies implemented and compared
- ✅ Privacy metrics calculated (k-anonymity)
- ✅ Statistical analysis (entropy, uniqueness, risk scores)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Default strategy balances accuracy and speed for general use
- Statistical-only useful for non-descriptive column names
- Custom dictionary provides highest accuracy for domain-specific data

**Strategy recommendations:**
- **Use Strategy 1** when: Standard attribute names with good descriptiveness
- **Use Strategy 2** when: Column names are non-descriptive or in foreign languages
- **Use Strategy 3** when: Domain-specific data (healthcare, finance, HR) requiring precision

**Next steps:**
- Test with your own datasets
- Customize dictionary for your domain
- Integrate with anonymization pipelines
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)